# Enformer Regulatory Activity Prediction

![Enformer Regulatory Activity Prediction](https://proto-bio.github.io/proto-assets/images/tool/enformer/hero.png)

Enformer is a deep learning model for predicting gene regulatory activity directly from DNA sequence, published by Avsec et al. (2021). It processes 196,608 bp input windows and outputs binned predictions across 896 spatial bins for thousands of regulatory tracks covering chromatin accessibility, histone modifications, and gene expression. The model supports both human and mouse output heads, and track selection lets you extract only the signals relevant to your analysis.

This notebook demonstrates how to predict regulatory activity from a DNA sequence and how to perform a variant-effect comparison by running reference and alternate allele windows through the model and computing log2 fold-change.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("enformer")
display_overview("enformer")
display_docs_section("enformer", "Background")

# Enformer

[Enformer](https://github.com/google-deepmind/deepmind-research/tree/master/enformer) is a deep learning model that predicts [gene expression](https://en.wikipedia.org/wiki/Gene_expression) and chromatin activity directly from DNA sequence, developed by Google DeepMind and Calico Life Sciences. It reads a fixed 196,608 base-pair input window and predicts activity across thousands of genomic measurement tracks, summarized over 896 output bins of 128 base pairs each. This toolkit exposes Enformer through a single registered tool that returns the predicted track activity for one or more sequences.

Enformer ([Avsec et al., 2021](https://doi.org/10.1038/s41592-021-01252-x)) is a neural network that predicts [gene expression](https://en.wikipedia.org/wiki/Gene_expression) and chromatin state from genomic DNA sequence. Its architecture combines convolutional layers with [transformer](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)) self-attention, which allows the model to integrate the influence of distal [cis-regulatory elements](https://en.wikipedia.org/wiki/Cis-regulatory_element) such as [enhancers](https://en.wikipedia.org/wiki/Enhancer_(genetics)) located up to 100 kilobases away from a [promoter](https://en.wikipedia.org/wiki/Promoter_(genetics)). The published work reports that this long-range modeling substantially improves gene expression prediction accuracy relative to earlier convolutional models, and that it yields more accurate predictions of the effect of genetic variants on expression for both natural variants and saturation mutagenesis measured by reporter assays.

Enformer predicts activity across 896 output bins, each summarizing 128 base pairs, for a large panel of functional genomics assays. The human output head covers 5,313 tracks spanning [chromatin accessibility](https://en.wikipedia.org/wiki/DNase_I_hypersensitive_site), [transcription factor](https://en.wikipedia.org/wiki/Transcription_factor) binding, [histone modifications](https://en.wikipedia.org/wiki/Histone), and [CAGE](https://en.wikipedia.org/wiki/Cap_analysis_gene_expression) expression measurements, and the mouse output head covers 1,643 tracks. Because the model maps sequence directly to these signals, it can be used to compare the predicted regulatory activity of alternative alleles at the same locus, which is the basis for its use in interpreting noncoding genetic variation. The published analysis additionally shows that Enformer learns enhancer-promoter relationships directly from the input sequence.

## Available tools

In [2]:
display_available_tools("enformer")

- **`run_enformer()`** — Gene expression and regulatory activity prediction using Enformer

### `run_enformer`

Enformer predicts regulatory activity from a 196,608 bp DNA input window, returning a signal matrix of shape [896, num_tracks] where each of the 896 spatial bins covers a 128 bp window. The `output_tracks` parameter selects which of the thousands of regulatory tracks to extract, and the `species` parameter switches between human and mouse output heads. A common use is variant-effect analysis: running the same input window with reference and alternate alleles and comparing the resulting prediction matrices to identify regulatory impact.

In [3]:
import numpy as np
from proto_tools import ENFORMER_CONTEXT, EnformerConfig, EnformerInput, run_enformer

In [4]:
# Display input docs
display_api_reference("enformer", "input", "run_enformer")

from proto_tools import NCBIEfetchConfig, NCBIEfetchInput, run_ncbi_efetch

# Fetch a real human genomic window on chr11 centred on the HBB locus, exactly the
# length Enformer requires, instead of a synthetic repeat.
_center = 5227430  # HBB midpoint, GRCh38 chr11 (NC_000011.10)
_fetch = run_ncbi_efetch(
    NCBIEfetchInput(
        db="nuccore", identifier="NC_000011.10", return_format="fasta",
        seq_start=_center - ENFORMER_CONTEXT // 2, seq_stop=_center + ENFORMER_CONTEXT // 2,
    ),
    NCBIEfetchConfig(),
)
sequence = _fetch.fasta_records[0].sequence.upper()[:ENFORMER_CONTEXT]
inputs = EnformerInput(sequences=[sequence])

**Input** — `EnformerInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[SequenceWindow]</code> | required | Sequence(s) to score; each a DNA window with optional target_range (a bare string is accepted) |

In [5]:
# Display config docs
display_api_reference("enformer", "config", "run_enformer")

config = EnformerConfig(
    output_tracks=[0, 1, 2],
    species="human",
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=False,
)

**Config** — `EnformerConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>output_tracks</code> | <code>list[int]</code> | <code>[0]</code> | Indices of Enformer output tracks to extract (5313 human / 1643 mouse) |
| <code>species</code> | <code>Literal['human', 'mouse']</code> | <code>'human'</code> | Species track head to use |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Number of sequences to process simultaneously on GPU |

In [6]:
# Run the prediction function
pred_result = run_enformer(inputs, config)

Running run_enformer [00:00]

In [7]:
display_api_reference("enformer", "output", "run_enformer")

print(f"Output bins: {len(pred_result.results[0].prediction)}")
print(f"Tracks per bin: {len(pred_result.results[0].prediction[0])}")

**Output** — `EnformerOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[EnformerPredictionResult]</code> | required | Per-sequence Enformer prediction results |
| <code>output_tracks</code> | <code>list[int]</code> | required | Track indices extracted from Enformer |
| <code>species</code> | <code>str</code> | required | Species used for prediction ('human' or 'mouse') |

**`EnformerPredictionResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | DNA sequence originally provided to the tool |
| <code>sequence_length</code> | <code>int</code> | required | Length of the provided DNA sequence |
| <code>prediction</code> | <code>list[list[float]]</code> | required | Predicted track activity per 128 bp bin (shape [896, num_tracks]) |
| <code>context_start</code> | <code>int</code> | required | 0-based start of the Enformer input window in sequence |
| <code>context_end</code> | <code>int</code> | required | 0-based exclusive end of the Enformer input window |
| <code>output_start</code> | <code>int</code> | required | 0-based start of the span covered by Enformer output bins |
| <code>output_end</code> | <code>int</code> | required | 0-based exclusive end of the Enformer output-bin span |
| <code>output_resolution</code> | <code>int</code> | <code>128</code> | Base pairs represented by each output bin |
| <code>target_start</code> | <code>int &#124; None</code> | <code>None</code> | Requested target start, if target_ranges was provided |
| <code>target_end</code> | <code>int &#124; None</code> | <code>None</code> | Requested target end, if target_ranges was provided |

Output bins: 896
Tracks per bin: 3


#### Variant-Effect Comparison

A common use of Enformer is comparing predictions between a reference and an alternate allele to identify regulatory variants. We introduce a single-nucleotide change at the center of the input window, run both sequences through the model, and compute log2 fold-change across all bins and tracks. Large absolute fold-change values indicate positions and tracks where the variant has a meaningful effect on predicted regulatory activity.

In [8]:
# Introduce a single-nucleotide substitution at the center of the input window
center = ENFORMER_CONTEXT // 2
ref_sequence = sequence
alt_base = "A" if ref_sequence[center] != "A" else "C"
alt_sequence = ref_sequence[:center] + alt_base + ref_sequence[center + 1:]

ref_result = run_enformer(EnformerInput(sequences=[ref_sequence]), config)
alt_result = run_enformer(EnformerInput(sequences=[alt_sequence]), config)

ref_pred = np.array(ref_result.results[0].prediction)
alt_pred = np.array(alt_result.results[0].prediction)
log2_fc = np.log2((alt_pred + 1e-6) / (ref_pred + 1e-6))
print(f"Max |log2 FC|: {np.abs(log2_fc).max():.4f}")

Running run_enformer [00:00]

Running run_enformer [00:00]

Max |log2 FC|: 0.0606


## Export Results

Prediction outputs can be saved to standard file formats for downstream analysis or integration with other computational pipelines. Supported formats include JSON and CSV.

In [9]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

pred_result.export(name="enformer_prediction", export_path=output_dir, file_format="json")
print(f"Prediction exported to {output_dir}/enformer_prediction.json")

ref_result.export(name="enformer_ref", export_path=output_dir, file_format="csv")
print(f"Reference summary exported to {output_dir}/enformer_ref.csv")

Prediction exported to example_output/enformer_prediction.json
Reference summary exported to example_output/enformer_ref.csv
